In [1]:
from utils import *
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch

In [3]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

In [4]:
def wordle_reward(guess: str, secret_word: str) -> int:
    if guess.upper() == secret_word.upper():
        return 1   # correct guess
    else:
        return 0   # incorrect guess

In [5]:
secret_word = "ROUND"

past_guesses = [
    GuessWithFeedback.from_secret(guess="CRANE", secret=secret_word),
    GuessWithFeedback.from_secret(guess="BLOND", secret=secret_word),
    GuessWithFeedback.from_secret(guess="POUND", secret=secret_word),
]
past_guesses

[CRANE → Feedback: C(x) R(-) A(x) N(✓) E(x),
 BLOND → Feedback: B(x) L(x) O(-) N(✓) D(✓),
 POUND → Feedback: P(x) O(✓) U(✓) N(✓) D(✓)]

In [6]:
response = generate(get_messages(past_guesses))[0]
guess = extract_guess(response)
reward = wordle_reward(guess, secret_word)

print(f"Guessed Word: {guess} -> Reward: {reward}")

Guessed Word: FOUND -> Reward: 0


In [7]:
def compute_advantages(rewards: list):
    rewards = np.array(rewards)
    
    # Compute the mean and standard deviation of the rewards
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards)

    # Avoid division by zero in case of zero variance (typically happens when all rewards are 0)
    # Note: In the GRPO implementation, we add 1e-4 to the std_reward to avoid division by zero
    if std_reward == 0:
        return [0] * len(rewards)

    # Divide by stddev of rewards to normalize range to 0
    advantages = (rewards - mean_reward) / std_reward
    return advantages.tolist()

In [8]:
rewards = [0.0, 0.2, 0.4, 0.5, 0.5, 0.6, 0.8, 1.0]
compute_advantages(rewards)

[-1.6903085094570331,
 -1.01418510567422,
 -0.33806170189140655,
 0.0,
 0.0,
 0.33806170189140655,
 1.0141851056742202,
 1.6903085094570331]

In [9]:
def render_guess_table(response, reward_fn):
    guesses = [extract_guess(guess) for guess in response]
    rewards = [reward_fn(guess, secret_word) for guess in guesses]
    print_guesses_table(guesses, rewards)

In [11]:
print(f"Secret: {secret_word}")
response = generate(get_messages(past_guesses), num_guesses=8)
render_guess_table(response, wordle_reward)

Secret: ROUND
+---------+---------+----------+-------------+
|   Index | Guess   |   Reward |   Advantage |
+=========+=========+==========+=============+
|       0 | FOUND   |        0 |    -0.57735 |
+---------+---------+----------+-------------+
|       1 | FOUND   |        0 |    -0.57735 |
+---------+---------+----------+-------------+
|       2 | FOUND   |        0 |    -0.57735 |
+---------+---------+----------+-------------+
|       3 | FOUND   |        0 |    -0.57735 |
+---------+---------+----------+-------------+
|       4 | FOUND   |        0 |    -0.57735 |
+---------+---------+----------+-------------+
|       5 | COUNT   |        0 |    -0.57735 |
+---------+---------+----------+-------------+
|       6 | ROUND   |        1 |     1.73205 |
+---------+---------+----------+-------------+
|       7 | ROUND   |        1 |     1.73205 |
+---------+---------+----------+-------------+


In [12]:
def wordle_reward_partial_credit(guess: str, secret_word: str) -> float:
    if len(guess) != len(secret_word):
        # no reward for having the wrong number of letters
        return 0.0
    
    valid_letters = set(secret_word)
    reward = 0.0
    for letter, secret_letter in zip(guess, secret_word):
        if letter == secret_letter:
            # right letter, right location
            reward += 0.2
        elif letter in valid_letters:
            # right letter, wrong location
            reward += 0.1
        else:
            # no reward
            pass
    return reward

In [13]:
print(f"Secret: {secret_word}")
response = generate(get_messages(past_guesses), num_guesses=8, temperature=0)
render_guess_table(response, wordle_reward_partial_credit)

Secret: ROUND
+---------+---------+----------+-------------+
|   Index | Guess   |   Reward |   Advantage |
+=========+=========+==========+=============+
|       0 | ROUND   |      1   |    1.29099  |
+---------+---------+----------+-------------+
|       1 | FOUND   |      0.8 |   -0.774597 |
+---------+---------+----------+-------------+
|       2 | ROUND   |      1   |    1.29099  |
+---------+---------+----------+-------------+
|       3 | FOUND   |      0.8 |   -0.774597 |
+---------+---------+----------+-------------+
|       4 | FOUND   |      0.8 |   -0.774597 |
+---------+---------+----------+-------------+
|       5 | FOUND   |      0.8 |   -0.774597 |
+---------+---------+----------+-------------+
|       6 | ROUND   |      1   |    1.29099  |
+---------+---------+----------+-------------+
|       7 | FOUND   |      0.8 |   -0.774597 |
+---------+---------+----------+-------------+


In [14]:
print(f"Secret: {secret_word}")
response = generate(get_messages(past_guesses), num_guesses=8, temperature=1.3)
render_guess_table(response, wordle_reward_partial_credit)

Secret: ROUND
+---------+---------+----------+-------------+
|   Index | Guess   |   Reward |   Advantage |
+=========+=========+==========+=============+
|       0 | FOUND   |      0.8 |    0.107211 |
+---------+---------+----------+-------------+
|       1 | TOUCH   |      0.4 |   -1.60817  |
+---------+---------+----------+-------------+
|       2 | FOUND   |      0.8 |    0.107211 |
+---------+---------+----------+-------------+
|       3 | ROUND   |      1   |    0.964901 |
+---------+---------+----------+-------------+
|       4 | ROUND   |      1   |    0.964901 |
+---------+---------+----------+-------------+
|       5 | HOUSE   |      0.4 |   -1.60817  |
+---------+---------+----------+-------------+
|       6 | ROUND   |      1   |    0.964901 |
+---------+---------+----------+-------------+
|       7 | FOUND   |      0.8 |    0.107211 |
+---------+---------+----------+-------------+


In [15]:
print(f"Secret: {secret_word}")
response = generate(get_messages(past_guesses), num_guesses=8, temperature=0.7)
render_guess_table(response, wordle_reward_partial_credit)

Secret: ROUND
+---------+---------+----------+-------------+
|   Index | Guess   |   Reward |   Advantage |
+=========+=========+==========+=============+
|       0 | COUNT   |      0.6 |   -1.50756  |
+---------+---------+----------+-------------+
|       1 | ROUND   |      1   |    0.904534 |
+---------+---------+----------+-------------+
|       2 | FOUND   |      0.8 |   -0.301511 |
+---------+---------+----------+-------------+
|       3 | ROUND   |      1   |    0.904534 |
+---------+---------+----------+-------------+
|       4 | ROUND   |      1   |    0.904534 |
+---------+---------+----------+-------------+
|       5 | ROUND   |      1   |    0.904534 |
+---------+---------+----------+-------------+
|       6 | FOUND   |      0.8 |   -0.301511 |
+---------+---------+----------+-------------+
|       7 | COULD   |      0.6 |   -1.50756  |
+---------+---------+----------+-------------+
